# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [1]:
import pandas as pd
PATH = 'https://raw.githubusercontent.com/Angela-Benedict/ds2002-fa26/refs/heads/main/notebooks/06-midterm-walmart/data/product_catalog.csv'
try:
    cat = pd.read_csv(PATH)
except FileNotFoundError:
    from io import StringIO
    cat = pd.read_csv(StringIO('sku,product_name,category\nPT-12,pop-tarts strawberry,Breakfast Foods\nSKU#459812,Strawberry Pop-Tart,Breakfast\nPOPTART-STRAW,Pop-Tarts Strawberry,Breakfast'))
cat.head()

,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack
4,WTR-24,BOTTLED WATER 24PK,Emergency Supplies,Bottled,2.64,Sara Lee,1 pack


### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [2]:
# TODO (hint: .str.lower().str.contains('pop'))
cat['product_name'] = cat['product_name'].str.lower()
matching_rows = cat[cat['product_name'].str.contains('pop')]
print('Count =',len(matching_rows))
matching_rows

Count = 4


,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,strawberry pop-tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,pop-tarts strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack


### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

In [3]:
other_pop = []
print(other_pop)
print('SKUs and corresponding names that all mean strawberry pop-tart are PT-12: pop-tarts strawberry, SKU#459812: strawberry pop-tart, POPTART-STRAW: pop-tarts strawberry, and PT_STRAWBERRY: pop-tarts strawberry.')
print('There are no pop-tart rows that do not contain strawberry, but any other flavor would stay separate.')

[]
SKUs and corresponding names that all mean strawberry pop-tart are PT-12: pop-tarts strawberry, SKU#459812: strawberry pop-tart, POPTART-STRAW: pop-tarts strawberry, and PT_STRAWBERRY: pop-tarts strawberry.
There are no pop-tart rows that do not contain strawberry, but any other flavor would stay separate.


**Pop-Tart rows that are a different product are not present but would stay separate if they were.

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [4]:
sku_mapping = {
    'PT-12': 'PT-12',
    'SKU#459812': 'PT-12',
    'POPTART-STRAW': 'PT-12',
    'PT_STRAWBERRY': 'PT-12',
}

cat['canonical_sku'] = (
    cat['sku']
    .map(sku_mapping)
    .fillna(cat['sku'])
)

### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [5]:
before = cat['category'].unique()
before_count = cat['category'].nunique()

# Strip whitespace and title-case
cat['category'] = cat['category'].str.strip().str.title()

after = cat['category'].unique()
after_count = cat['category'].nunique()

print('Before:')
print(before, before_count)
print()

print('After:')
print(after, after_count)
print()

print('Pairs that may represent the same category:')
print('Breakfast Foods / Breakfast')
print('Emergency Supplies / Emergency')
print('Alcoholic Beverages / Alcohol')
print('Groceries / Grocery')
print('Baby / Baby Products')

Before:
['Breakfast Foods' 'Breakfast' 'breakfast' 'Emergency Supplies'
 'Emergency' 'EMERGENCY' 'Alcoholic Beverages' 'ALCOHOL' 'ENTERTAINMENT'
 'entertainment' 'Electronics' 'electronics' 'Groceries' 'GROCERY'
 'Grocery' 'grocery' 'health' 'ELECTRONICS' 'snacks' 'Snacks' 'household'
 'HOUSEHOLD' 'Household' 'baby' 'Baby Products' 'Baby' 'emergency'] 27

After:
['Breakfast Foods' 'Breakfast' 'Emergency Supplies' 'Emergency'
 'Alcoholic Beverages' 'Alcohol' 'Entertainment' 'Electronics' 'Groceries'
 'Grocery' 'Health' 'Snacks' 'Household' 'Baby' 'Baby Products'] 15

Pairs that may represent the same category:
Breakfast Foods / Breakfast
Emergency Supplies / Emergency
Alcoholic Beverages / Alcohol
Groceries / Grocery
Baby / Baby Products


### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [6]:
import string

def normalize_product_name(name):
    if pd.isna(name):
        return None

    words = name.lower().translate(
        str.maketrans("", "", string.punctuation)
    ).split()

    return " ".join(sorted(words))

cat['normalized_name'] = cat['product_name'].apply(normalize_product_name)

clean_cat = cat.dropna(subset=['normalized_name', 'sku']).copy()

clean_cat['sku_counts'] = (
    clean_cat.groupby('normalized_name')['sku']
    .transform('nunique')
)

candidate_list = (
    clean_cat[clean_cat['sku_counts'] > 1]
    .sort_values('normalized_name')
)
print(candidate_list[['sku', 'product_name', 'normalized_name', 'sku_counts']])

              sku           product_name      normalized_name  sku_counts
12        BEER-24   light beer (24-pack)    24pack beer light           2
13       LTBEER24   light beer (24-pack)    24pack beer light           2
8         BAT-AA4    aa batteries 4-pack   4pack aa batteries           2
9     BATT_AA_4PK    aa batteries 4-pack   4pack aa batteries           2
14        BG-MONO    monopoly board game  board game monopoly           3
16      GAME-MONO  board game - monopoly  board game monopoly           3
15    MONOPOLY-01    monopoly board game  board game monopoly           3
23       BREAD-WH            white bread          bread white           2
24     WHITEBREAD            white bread          bread white           2
26     PNUTBTR-16          peanut butter        butter peanut           2
25          PB-16          peanut butter        butter peanut           2
28    CANDLE-EMRG      emergency candles    candles emergency           2
27        CNDL-EM    candles (emergenc

### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [7]:
rows_before = len(cat)

cat['canonical_sku'] = cat['sku'].map(sku_mapping).fillna(cat['sku'])

assert cat['canonical_sku'].notna().all(), 'Null values found in canonical_sku'
assert len(cat) == rows_before, 'Row count changed during consolidation'

### Write-up

**a)** Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.

**b)** Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.

a) 1: Canned soup/ soup - canned under Groceries and Grocery categories appear fragmented. I would group them all under "Grocery" with the white bread and peanut butter.

2: baby diapers - size 3, diapers sz 3 under Baby Products and Baby. I would group them under "Baby" with infant/baby formula.


b) Consolidation becomes a business decision when it's unclear what category a similarly-named product is under. For example, as a data-cleaner I'm not sure if emergency candles are the same product as candles(emergency) because emergency candles could be used for store emergencies but the other candles could just be regular candles used for a shopper's needs/other emergency. I would ask a store manager/employee that stocks regularily to understand the difference between the products before I categorize/group them.